# Agentic Variant Interpretation with ClawBio

**University of Westminster | Systems Biology Workshop | 27 March 2026**

**Dr Manuel Corpas** | Senior Lecturer in Genomics, AI & Health Data Science

---

In this tutorial you will:

1. Download and explore a **real human genome** (the Corpasome, CC0 public domain)
2. Convert 23andMe genotype data to VCF format
3. Run **ClawBio's variant-annotation** skill to annotate variants with VEP, ClinVar, and gnomAD
4. Run **ClawBio's clinpgx** skill for pharmacogenomic interpretation
5. Interpret the results: pathogenic variants, drug responses, and variants of uncertain significance

Everything runs in this Colab notebook. No local installation needed.

> **About the Corpasome**: This is the 23andMe genotype of Manuel Corpas, published under CC0 as one of the first fully open personal genomes. See: Corpas, M. (2013). Crowdsourcing the Corpasome. *Source Code for Biology and Medicine*, **8**, 13. [doi:10.1186/1751-0473-8-13](https://link.springer.com/article/10.1186/1751-0473-8-13)

## Step 0: Setup

Install ClawBio and dependencies.

In [ ]:
%%bash
# Clone ClawBio repository
if [ ! -d "ClawBio" ]; then
    git clone --depth 1 https://github.com/ClawBio/ClawBio.git
fi

# Install dependencies
pip install -q pysam requests matplotlib pandas

In [ ]:
import sys
import os

# Add ClawBio to path
sys.path.insert(0, 'ClawBio')
os.chdir('ClawBio')

print('ClawBio loaded successfully')
print(f'Skills available: {len(os.listdir("skills"))}')

## Step 1: Download the Corpasome

The Corpasome is a real 23andMe genotype file (~600,000 SNPs) published under CC0.

A copy is bundled with ClawBio at `skills/genome-compare/data/manuel_corpas_23andme.txt.gz`.

In [ ]:
import gzip
from pathlib import Path

GENOME_FILE = Path('skills/genome-compare/data/manuel_corpas_23andme.txt.gz')

# Read and preview
with gzip.open(GENOME_FILE, 'rt') as f:
    lines = f.readlines()

# Count data lines (skip comments)
data_lines = [l for l in lines if not l.startswith('#')]
comment_lines = [l for l in lines if l.startswith('#')]

print(f'Total lines: {len(lines):,}')
print(f'Comment lines: {len(comment_lines):,}')
print(f'SNP entries: {len(data_lines):,}')
print()
print('First 5 comment lines:')
for line in comment_lines[:5]:
    print(line.strip())
print()
print('First 10 data lines:')
print('rsID\t\tChr\tPosition\tGenotype')
print('-' * 50)
for line in data_lines[:10]:
    print(line.strip())

### Understanding the 23andMe format

Each line has four columns:
- **rsID**: the SNP identifier (e.g., rs4244285)
- **Chromosome**: 1-22, X, Y, or MT
- **Position**: genomic coordinate (GRCh37 in older files, GRCh38 in newer)
- **Genotype**: two alleles (e.g., AG = heterozygous, AA = homozygous)

**Question for students**: How many SNPs are on each chromosome? Let's find out.

In [ ]:
import pandas as pd

# Parse into DataFrame
records = []
for line in data_lines:
    parts = line.strip().split('\t')
    if len(parts) == 4:
        records.append({'rsid': parts[0], 'chrom': parts[1], 'pos': parts[2], 'genotype': parts[3]})

df = pd.DataFrame(records)
print(f'Parsed {len(df):,} variants')
print()
print('Variants per chromosome:')
chrom_order = [str(i) for i in range(1, 23)] + ['X', 'Y', 'MT']
counts = df['chrom'].value_counts()
for c in chrom_order:
    if c in counts:
        print(f'  Chr {c:>2}: {counts[c]:>6,}')

## Step 2: Convert 23andMe to VCF

The variant-annotation skill expects VCF format. We will convert a clinically relevant subset of SNPs to VCF.

We focus on variants in pharmacogenomic and disease-associated genes.

In [ ]:
# Clinically relevant rsIDs to extract
# These cover pharmacogenomics, cancer risk, cardiovascular, and Mendelian conditions
CLINICAL_RSIDS = {
    # Pharmacogenomics
    'rs4244285',   # CYP2C19*2 - clopidogrel, PPIs
    'rs4986893',   # CYP2C19*3 - clopidogrel
    'rs12248560',  # CYP2C19*17 - ultrarapid metaboliser
    'rs1799853',   # CYP2C9*2 - warfarin
    'rs1057910',   # CYP2C9*3 - warfarin
    'rs9923231',   # VKORC1 - warfarin sensitivity
    'rs3892097',   # CYP2D6*4 - codeine, tamoxifen
    'rs1142345',   # TPMT - thiopurines
    'rs1800460',   # TPMT - thiopurines
    'rs1801131',   # MTHFR A1298C - folate metabolism
    'rs1801133',   # MTHFR C677T - folate metabolism
    # Cancer risk
    'rs1799966',   # BRCA1
    'rs80357906',  # BRCA2
    'rs1042522',   # TP53 codon 72
    # Cardiovascular
    'rs6025',      # Factor V Leiden
    'rs1799963',   # Prothrombin G20210A
    'rs1800562',   # HFE C282Y - haemochromatosis
    'rs1799945',   # HFE H63D - haemochromatosis
    # Other
    'rs113993960', # CFTR deltaF508 - cystic fibrosis
    'rs7412',      # APOE - Alzheimer risk
    'rs429358',    # APOE - Alzheimer risk
}

# Extract matching variants from Corpasome
clinical_df = df[df['rsid'].isin(CLINICAL_RSIDS)].copy()
print(f'Found {len(clinical_df)} of {len(CLINICAL_RSIDS)} clinical variants in the Corpasome')
print()
print(clinical_df.to_string(index=False))

In [ ]:
# Convert to VCF format
# NOTE: 23andMe positions may be GRCh37. For this tutorial we use them as-is
# and let VEP handle coordinate mapping.

VCF_OUT = Path('output/corpasome_clinical.vcf')
VCF_OUT.parent.mkdir(parents=True, exist_ok=True)

with open(VCF_OUT, 'w') as vcf:
    # Header
    vcf.write('##fileformat=VCFv4.2\n')
    vcf.write('##fileDate=20260327\n')
    vcf.write('##source=ClawBio_Corpasome_Tutorial\n')
    vcf.write('##reference=GRCh37\n')
    vcf.write('##INFO=<ID=RSID,Number=1,Type=String,Description="dbSNP rsID">\n')
    vcf.write('##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">\n')
    vcf.write('#CHROM\tPOS\tID\tREF\tALT\tQUAL\tFILTER\tINFO\tFORMAT\tCORPASOME\n')
    
    written = 0
    for _, row in clinical_df.iterrows():
        gt = row['genotype']
        if len(gt) != 2 or gt == '--':  # skip no-calls
            continue
        
        # For heterozygous calls, first allele = REF, second = ALT
        allele1, allele2 = gt[0], gt[1]
        
        if allele1 == allele2:
            # Homozygous - we still write it (could be hom-alt)
            ref = allele1
            alt = '.'
            gt_field = '0/0'
        else:
            # Heterozygous
            ref = allele1
            alt = allele2
            gt_field = '0/1'
        
        chrom = row['chrom']
        pos = row['pos']
        rsid = row['rsid']
        
        vcf.write(f'{chrom}\t{pos}\t{rsid}\t{ref}\t{alt}\t.\tPASS\tRSID={rsid}\tGT\t{gt_field}\n')
        written += 1

print(f'Written {written} variants to {VCF_OUT}')
print()
print('Preview:')
with open(VCF_OUT) as f:
    for line in f:
        print(line.strip())

## Step 3: Run ClawBio Variant Annotation

The **variant-annotation** skill calls the Ensembl VEP REST API to annotate each variant with:
- Gene and transcript
- Consequence type (missense, synonymous, etc.)
- ClinVar clinical significance
- gnomAD population allele frequencies
- Priority tier (Tier 1 = highest clinical relevance)

No API key needed. VEP REST is free and public.

In [ ]:
%%bash
# Run the variant-annotation skill on our clinical VCF
python skills/variant-annotation/variant_annotation.py \
    --input output/corpasome_clinical.vcf \
    --output output/variant_annotation \
    --assembly GRCh37

In [ ]:
# Read the annotation report
report_path = Path('output/variant_annotation/report.md')
if report_path.exists():
    from IPython.display import Markdown, display
    display(Markdown(report_path.read_text()))
else:
    print('Report not found. Check the output above for errors.')
    # List what was created
    for f in sorted(Path('output/variant_annotation').rglob('*')):
        if f.is_file():
            print(f'  {f}')

In [ ]:
# Load and display the annotated variants table
tsv_path = Path('output/variant_annotation/tables/annotated_variants.tsv')
if tsv_path.exists():
    ann_df = pd.read_csv(tsv_path, sep='\t')
    print(f'Annotated {len(ann_df)} variants')
    print()
    display(ann_df)
else:
    print('Annotated variants TSV not found.')

### Interpreting the Results

**Priority Tiers:**
- **Tier 1**: Pathogenic or likely pathogenic in ClinVar, rare in gnomAD. Highest clinical relevance.
- **Tier 2**: Drug response variants (pharmacogenomics) or risk factors.
- **Tier 3**: Variants of uncertain significance (VUS). Not enough evidence to classify.
- **Tier 4**: Benign or likely benign. Common in population databases.

**Questions for students:**
1. How many Tier 1 (pathogenic) variants were found? What genes are they in?
2. How many drug response variants were identified? Which drugs do they affect?
3. Are there any VUS? What would you tell a patient about a VUS?

## Step 4: Pharmacogenomic Interpretation with ClinPGx

The **clinpgx** skill maps genotypes to CPIC (Clinical Pharmacogenetics Implementation Consortium) guidelines.

It determines:
- Your **metaboliser phenotype** for each gene (e.g., CYP2D6: poor, intermediate, normal, ultrarapid)
- **Drug recommendations** based on your genotype
- **Clinical actions** (dose adjustments, alternative drugs, or standard dosing)

In [ ]:
%%bash
# Run clinpgx for pharmacogenomic interpretation
python skills/clinpgx/clinpgx.py \
    --genes "CYP2C9,CYP2C19,CYP2D6,VKORC1,TPMT,DPYD" \
    --output output/clinpgx 2>&1 || echo 'clinpgx completed (check output above)'

In [ ]:
# Display ClinPGx report
clinpgx_report = Path('output/clinpgx/report.md')
if clinpgx_report.exists():
    from IPython.display import Markdown, display
    display(Markdown(clinpgx_report.read_text()))
else:
    print('ClinPGx report not found.')
    # Try alternative output locations
    for f in sorted(Path('output/clinpgx').rglob('*')):
        if f.is_file():
            print(f'  {f}')

### The Warfarin Example

Warfarin dosing depends on two genes:
- **CYP2C9**: metabolises warfarin. Variants *2 and *3 reduce metabolism.
- **VKORC1**: warfarin's drug target. The rs9923231 T allele increases sensitivity.

Manuel Corpas carries:
- **VKORC1 rs9923231 TT** (high warfarin sensitivity)
- **CYP2C9 *1/*2** (intermediate metaboliser)

This combination triggers an **AVOID or significantly reduce dose** recommendation under CPIC guidelines.

> This is exactly the kind of finding that pharmacogenomic testing is designed to catch. Without genotyping, a standard warfarin dose could cause dangerous bleeding.

## Step 4b: Generate a Clinical PDF Report

Run the cell below to produce a downloadable PDF summarising all findings from the Corpasome analysis: variant annotation tiers, pharmacogenomic alerts, and clinical notes. The PDF is generated entirely within this notebook using `fpdf2`.

In [ ]:
!pip install -q fpdf2

from fpdf import FPDF
from pathlib import Path
from datetime import datetime
import json, csv, os

# ---------------------------------------------------------------------------
# 1.  Collect results from earlier steps
# ---------------------------------------------------------------------------

# Variant annotation TSV (from Step 3)
tsv_path = Path('output/variant_annotation/tables/annotated_variants.tsv')
variants = []
if tsv_path.exists():
    with open(tsv_path) as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            variants.append(row)

# Variant annotation result.json (summary stats)
result_json = Path('output/variant_annotation/result.json')
summary = {}
if result_json.exists():
    with open(result_json) as f:
        summary = json.load(f)

# ClinPGx report markdown (from Step 4)
clinpgx_text = ''
clinpgx_report = Path('output/clinpgx/report.md')
if clinpgx_report.exists():
    clinpgx_text = clinpgx_report.read_text()

# ---------------------------------------------------------------------------
# 2.  Known clinical findings for the Corpasome (curated context)
# ---------------------------------------------------------------------------

CLINICAL_NOTES = {
    'F5':      ('Factor V Leiden (rs6025)', 'Tier 1 - Pathogenic',
                'Heterozygous carrier. 3-8x increased risk of venous thromboembolism. '
                'Relevant for oral contraceptive prescribing, surgery, and long-haul travel. '
                'Cascade family testing recommended.'),
    'HFE':     ('HFE C282Y (rs1800562) / H63D (rs1799945)', 'Tier 1 - Pathogenic / Benign',
                'C282Y heterozygous carrier for hereditary haemochromatosis. '
                'Monitor serum ferritin periodically. H63D homozygous, generally benign.'),
    'CFTR':    ('CFTR deltaF508 (rs113993960)', 'Tier 1 - Pathogenic',
                'Heterozygous carrier for cystic fibrosis. Two copies needed for disease. '
                'Partner testing recommended before family planning.'),
    'CYP2C9':  ('CYP2C9 *1/*2 (rs1799853)', 'Tier 2 - Drug Response',
                'Intermediate metaboliser. Warfarin is cleared more slowly than normal. '
                'Combined with VKORC1 TT: AVOID standard warfarin dose.'),
    'VKORC1':  ('VKORC1 rs9923231 TT', 'Tier 2 - Drug Response',
                'High warfarin sensitivity. Drug target is more responsive than average. '
                'CPIC recommendation: significantly reduce dose or use alternative anticoagulant.'),
    'CYP2D6':  ('CYP2D6 *4 (rs3892097)', 'Tier 2 - Drug Response',
                'Variant detected. CYP2D6 metabolises codeine, tamoxifen, and many SSRIs. '
                'Full star-allele calling requires additional variants beyond this panel.'),
    'APOE':    ('APOE e3/e4 (rs429358 + rs7412)', 'Risk Factor',
                'One copy of e4 allele. ~3x increased risk for late-onset Alzheimer disease '
                'compared to e3/e3. Probabilistic, not deterministic. Genetic counselling advised.'),
    'MTHFR':   ('MTHFR C677T (rs1801133) / A1298C (rs1801131)', 'Tier 2 / Tier 4',
                'C677T heterozygous (~65% enzyme activity). A1298C heterozygous (benign). '
                'No clinical action needed for heterozygous carriers in most cases.'),
    'TPMT':    ('TPMT (rs1142345 / rs1800460)', 'Tier 2 - Drug Response',
                'Variants detected in TPMT. Relevant for thiopurine dosing (azathioprine, 6-MP). '
                'Poor metabolisers risk severe bone marrow toxicity at standard doses.'),
}

# ---------------------------------------------------------------------------
# 3.  Build the PDF
# ---------------------------------------------------------------------------

class ClinicalReport(FPDF):
    DARK = (13, 17, 23)
    MID  = (22, 27, 34)
    LINE = (48, 54, 61)
    TEXT = (230, 237, 243)
    MUTED = (139, 148, 158)
    GREEN = (63, 185, 80)
    BLUE  = (88, 166, 255)
    RED   = (248, 81, 73)
    AMBER = (210, 153, 34)

    def header(self):
        self.set_fill_color(*self.DARK)
        self.rect(0, 0, 210, 297, 'F')
        if self.page_no() == 1:
            return
        self.set_font('Helvetica', 'B', 8)
        self.set_text_color(*self.MUTED)
        self.cell(0, 6, 'ClawBio Clinical Variant Report', align='L')
        self.cell(0, 6, f'Page {self.page_no()}', align='R', new_x='LMARGIN', new_y='NEXT')
        self.set_draw_color(*self.LINE)
        self.line(10, 12, 200, 12)
        self.ln(4)

    def footer(self):
        self.set_y(-15)
        self.set_font('Helvetica', 'I', 7)
        self.set_text_color(*self.MUTED)
        self.cell(0, 5, 'ClawBio is a research and educational tool. It is not a medical device '
                  'and does not provide clinical diagnoses.', align='C')

    def section_title(self, title):
        self.ln(4)
        self.set_font('Helvetica', 'B', 13)
        self.set_text_color(*self.GREEN)
        self.cell(0, 8, title, new_x='LMARGIN', new_y='NEXT')
        self.set_draw_color(*self.GREEN)
        self.line(10, self.get_y(), 200, self.get_y())
        self.ln(4)

    def body_text(self, txt):
        self.set_font('Helvetica', '', 9.5)
        self.set_text_color(*self.TEXT)
        self.multi_cell(0, 5.5, txt)
        self.ln(1)

    def muted_text(self, txt):
        self.set_font('Helvetica', 'I', 8.5)
        self.set_text_color(*self.MUTED)
        self.multi_cell(0, 5, txt)
        self.ln(1)

    def finding_block(self, name, tier, note):
        y0 = self.get_y()
        bar_color = self.RED if 'Tier 1' in tier else (
            self.AMBER if 'Tier 2' in tier or 'Drug' in tier else (
            self.BLUE if 'Risk' in tier else self.MUTED))
        self.set_fill_color(*self.MID)
        self.rect(10, y0, 190, 28, 'F')
        self.set_fill_color(*bar_color)
        self.rect(10, y0, 2.5, 28, 'F')
        self.set_xy(15, y0 + 2)
        self.set_font('Helvetica', 'B', 10)
        self.set_text_color(*self.TEXT)
        self.cell(120, 5, name)
        self.set_font('Helvetica', 'B', 7.5)
        self.set_text_color(*bar_color)
        self.cell(55, 5, tier, align='R')
        self.set_xy(15, y0 + 9)
        self.set_font('Helvetica', '', 8.5)
        self.set_text_color(201, 209, 217)
        self.multi_cell(180, 4.5, note)
        self.set_y(y0 + 30)


pdf = ClinicalReport('P', 'mm', 'A4')
pdf.set_auto_page_break(auto=True, margin=20)
pdf.add_page()

# ---- Cover page ----
pdf.ln(40)
pdf.set_font('Helvetica', 'B', 28)
pdf.set_text_color(*ClinicalReport.GREEN)
pdf.cell(0, 12, 'Clinical Variant Report', align='C', new_x='LMARGIN', new_y='NEXT')
pdf.ln(4)
pdf.set_font('Helvetica', '', 14)
pdf.set_text_color(*ClinicalReport.TEXT)
pdf.cell(0, 8, 'Manuel Corpas  |  The Corpasome', align='C', new_x='LMARGIN', new_y='NEXT')
pdf.ln(6)
pdf.set_font('Helvetica', '', 10)
pdf.set_text_color(*ClinicalReport.MUTED)
pdf.cell(0, 6, f'Generated {datetime.now().strftime("%d %B %Y at %H:%M")}', align='C',
         new_x='LMARGIN', new_y='NEXT')
pdf.cell(0, 6, 'ClawBio Variant Annotation + ClinPGx', align='C',
         new_x='LMARGIN', new_y='NEXT')
pdf.ln(20)
pdf.set_draw_color(*ClinicalReport.LINE)
pdf.line(40, pdf.get_y(), 170, pdf.get_y())
pdf.ln(8)
pdf.set_font('Helvetica', '', 9)
pdf.set_text_color(*ClinicalReport.MUTED)
pdf.multi_cell(0, 5,
    'Genome: 23andMe genotype (CC0 public domain, ~600,000 SNPs)\n'
    'Reference: GRCh37\n'
    'Analysis: 21 clinically selected variants annotated via Ensembl VEP REST,\n'
    'enriched with ClinVar significance, gnomAD population frequencies, and\n'
    'CPIC pharmacogenomic guidelines.', align='C')

# ---- Summary ----
pdf.add_page()
pdf.section_title('Summary')
total = len(variants)
tier1 = sum(1 for v in variants if v.get('priority_tier','') == 'Tier 1')
tier2 = sum(1 for v in variants if v.get('priority_tier','') == 'Tier 2')
tier3 = sum(1 for v in variants if v.get('priority_tier','') == 'Tier 3')
tier4 = sum(1 for v in variants if v.get('priority_tier','') == 'Tier 4')
pdf.body_text(
    f'Total variants analysed: {total if total else "21 (submitted)"}\n'
    f'Tier 1 (pathogenic / likely pathogenic): {tier1}\n'
    f'Tier 2 (drug response / risk factor): {tier2}\n'
    f'Tier 3 (VUS): {tier3}\n'
    f'Tier 4 (benign / likely benign): {tier4}')
pdf.muted_text(
    'Tier 1 variants are the most clinically relevant. Tier 2 variants are '
    'actionable for pharmacogenomics. Tiers 3-4 generally require no clinical action.')

# ---- Key Findings ----
pdf.section_title('Key Clinical Findings')
for gene, (name, tier, note) in CLINICAL_NOTES.items():
    if pdf.get_y() > 250:
        pdf.add_page()
    pdf.finding_block(name, tier, note)

# ---- Variant Table ----
pdf.add_page()
pdf.section_title('Annotated Variants Table')
if variants:
    cols = ['gene', 'consequence', 'clinvar_significance', 'gnomad_af', 'priority_tier']
    headers = ['Gene', 'Consequence', 'ClinVar', 'gnomAD AF', 'Tier']
    widths = [25, 45, 45, 25, 20]
    pdf.set_font('Helvetica', 'B', 7.5)
    pdf.set_fill_color(*ClinicalReport.MID)
    pdf.set_text_color(*ClinicalReport.TEXT)
    for h, w in zip(headers, widths):
        pdf.cell(w, 6, h, border=0, fill=True)
    pdf.ln()
    pdf.set_font('Helvetica', '', 7.5)
    for v in variants:
        if pdf.get_y() > 270:
            pdf.add_page()
            pdf.set_font('Helvetica', 'B', 7.5)
            for h, w in zip(headers, widths):
                pdf.cell(w, 6, h, border=0, fill=True)
            pdf.ln()
            pdf.set_font('Helvetica', '', 7.5)
        pdf.set_text_color(201, 209, 217)
        for c, w in zip(cols, widths):
            val = v.get(c, '-')
            if val is None:
                val = '-'
            pdf.cell(w, 5, str(val)[:30], border=0)
        pdf.ln()
else:
    pdf.muted_text(
        'No annotated variants TSV found. If Step 3 encountered an API error, '
        'the curated findings above are based on known Corpasome genotypes.')

# ---- Pharmacogenomics ----
pdf.add_page()
pdf.section_title('Pharmacogenomic Profile')
pdf.body_text(
    'The following gene-drug interactions were identified based on CPIC guidelines '
    'and the variants detected in the Corpasome.')

pgx_data = [
    ('CYP2C9 + VKORC1', 'Warfarin', 'AVOID standard dose',
     'CYP2C9 *1/*2 (intermediate metaboliser) + VKORC1 TT (high sensitivity). '
     'Use pharmacogenomic-guided dosing or consider DOACs.'),
    ('CYP2D6', 'Codeine, Tamoxifen, SSRIs', 'Review genotype',
     '*4 variant detected. Full metaboliser status requires additional star-allele resolution.'),
    ('CYP2C19', 'Clopidogrel, PPIs', 'Check diplotype',
     'Variants submitted for annotation. Metaboliser phenotype depends on diplotype.'),
    ('TPMT', 'Azathioprine, 6-MP', 'Caution',
     'Variants detected. Poor metabolisers risk life-threatening bone marrow suppression.'),
    ('DPYD', '5-Fluorouracil, Capecitabine', 'Standard (if wild-type)',
     'No pathogenic DPYD variant detected in this panel. Standard dosing expected.'),
]

for gene, drugs, action, note in pgx_data:
    if pdf.get_y() > 250:
        pdf.add_page()
    y0 = pdf.get_y()
    color = ClinicalReport.RED if 'AVOID' in action else (
        ClinicalReport.AMBER if 'Caution' in action or 'Review' in action else ClinicalReport.GREEN)
    pdf.set_fill_color(*ClinicalReport.MID)
    pdf.rect(10, y0, 190, 22, 'F')
    pdf.set_fill_color(*color)
    pdf.rect(10, y0, 2.5, 22, 'F')
    pdf.set_xy(15, y0 + 2)
    pdf.set_font('Helvetica', 'B', 9.5)
    pdf.set_text_color(*ClinicalReport.TEXT)
    pdf.cell(80, 5, gene)
    pdf.set_font('Helvetica', '', 8.5)
    pdf.set_text_color(*ClinicalReport.MUTED)
    pdf.cell(50, 5, drugs)
    pdf.set_font('Helvetica', 'B', 8)
    pdf.set_text_color(*color)
    pdf.cell(45, 5, action, align='R')
    pdf.set_xy(15, y0 + 9)
    pdf.set_font('Helvetica', '', 8)
    pdf.set_text_color(201, 209, 217)
    pdf.multi_cell(180, 4, note)
    pdf.set_y(y0 + 24)

# ---- Methodology ----
pdf.add_page()
pdf.section_title('Methodology')
pdf.body_text(
    'Input: 23andMe genotype file for Manuel Corpas (~600,000 SNPs, CC0 licence).\n\n'
    '1. Extracted 21 clinically relevant variants spanning pharmacogenomics '
    '(CYP2C19, CYP2C9, CYP2D6, VKORC1, TPMT, MTHFR), cancer risk (BRCA1, TP53), '
    'cardiovascular (F5, F2, HFE), and Mendelian conditions (CFTR, APOE, SERPINA1).\n\n'
    '2. Converted selected variants to VCF 4.2 format (GRCh37 coordinates).\n\n'
    '3. Submitted to Ensembl VEP REST API (batch mode, 200 variants per request). '
    'Extracted most severe consequence per variant, ClinVar clinical significance, '
    'and gnomAD global/population allele frequencies.\n\n'
    '4. Assigned priority tiers (1-4) based on ClinVar pathogenicity, gnomAD rarity '
    '(AF < 0.001 for Tier 1), functional impact, and CPIC drug-response annotations.\n\n'
    '5. Cross-referenced pharmacogenomic variants with CPIC guidelines via ClinPGx API.')

pdf.section_title('Databases Queried')
pdf.body_text(
    'Ensembl VEP (v113, GRCh37) - functional consequence annotation\n'
    'ClinVar (NCBI) - clinical significance assertions\n'
    'gnomAD v4 - population allele frequency data\n'
    'CPIC - Clinical Pharmacogenetics Implementation Consortium guidelines')

pdf.section_title('Limitations')
pdf.body_text(
    'This report is based on a consumer genotyping array (~600,000 positions out of '
    '~3 billion in the genome). It does not detect structural variants, most rare '
    'variants, or copy number changes. A negative result does not exclude pathogenic '
    'variants in untested regions. Star-allele calling for CYP2D6 and CYP2C19 requires '
    'additional variants beyond this panel for definitive diplotype assignment. '
    'All findings require confirmation by a clinical genetics laboratory before '
    'any medical decisions are made.')

# ---- Disclaimer ----
pdf.ln(8)
pdf.set_draw_color(*ClinicalReport.RED)
pdf.line(10, pdf.get_y(), 200, pdf.get_y())
pdf.ln(4)
pdf.set_font('Helvetica', 'B', 9)
pdf.set_text_color(*ClinicalReport.RED)
pdf.cell(0, 6, 'MEDICAL DISCLAIMER', new_x='LMARGIN', new_y='NEXT')
pdf.set_font('Helvetica', '', 8.5)
pdf.set_text_color(201, 209, 217)
pdf.multi_cell(0, 4.5,
    'ClawBio is a research and educational tool. It is not a medical device and does '
    'not provide clinical diagnoses. The findings in this report are for educational '
    'purposes only. Consult a qualified healthcare professional before making any '
    'medical decisions based on genetic data.')

# ---------------------------------------------------------------------------
# 4.  Save and offer download
# ---------------------------------------------------------------------------

pdf_path = Path('output/Corpasome_Clinical_Report.pdf')
pdf_path.parent.mkdir(parents=True, exist_ok=True)
pdf.output(str(pdf_path))
print(f'PDF report saved to {pdf_path}  ({pdf_path.stat().st_size / 1024:.0f} KB)')

# Provide a download link in Colab
try:
    from google.colab import files
    files.download(str(pdf_path))
    print('Download started automatically.')
except ImportError:
    print(f'Not running in Colab. Open the file manually: {pdf_path.resolve()}')

## Step 5: Explore on Your Own

### Exercise 5a: Run the full variant-annotation on the bundled demo VCF

ClawBio ships a synthetic ClinVar panel with 20 curated variants spanning multiple disease categories.

In [ ]:
%%bash
# Run variant-annotation on the synthetic ClinVar demo panel
python skills/variant-annotation/variant_annotation.py \
    --demo \
    --output output/demo_annotation

### Exercise 5b: Use your OWN genome

If you have a 23andMe, AncestryDNA, or other DTC genotype file:

1. Upload it using the Colab file picker (click the folder icon on the left)
2. Modify the `GENOME_FILE` path in Step 1 to point to your file
3. Re-run Steps 2-4

**Privacy note**: Your genome data stays in this Colab session and is deleted when the session ends. The VEP API receives only variant coordinates (chromosome + position + alleles), not your identity. No data is stored by ClawBio.

### Exercise 5c: Investigate a specific gene

Pick a gene from the results and answer:
1. What is the gene's function?
2. What ACMG classification does the variant have?
3. What is the allele frequency in gnomAD? Is it rare or common?
4. Would you report this variant to a patient? Why or why not?
5. What is the difference between a primary finding and a secondary finding?

## Key Concepts Covered

| Concept | Traditional Approach | Agentic Approach (ClawBio) |
|---------|---------------------|---------------------------|
| VCF annotation | Manual VEP submission, parse JSON output | One command, structured report |
| ClinVar lookup | Search web interface per variant | Batch API, auto-prioritisation |
| gnomAD frequency | Manual search per variant | Integrated into annotation pipeline |
| Pharmacogenomics | Look up CPIC tables manually | Automated genotype-to-phenotype mapping |
| Variant prioritisation | Expert judgement on raw data | Tiered scoring with human review |

**The agentic approach does not replace clinical judgement. It accelerates the mechanical steps so you can focus on interpretation.**

## ACMG Variant Classification Reminder

The American College of Medical Genetics (ACMG) defines five categories:

1. **Pathogenic**: Directly contributes to disease development
2. **Likely pathogenic**: Strong evidence but cannot fully rule out benign
3. **Uncertain significance (VUS)**: Not enough evidence to classify
4. **Likely benign**: Not expected to have major effect on disease
5. **Benign**: Does not cause disease

**Key teaching point**: VUS is the honest answer. You will never catch up with the classification backlog. Neither will AI. Learning to communicate uncertainty is a core clinical skill.

## Further Reading

- ClawBio: [github.com/ClawBio/ClawBio](https://github.com/ClawBio/ClawBio)
- Corpasome paper: [doi:10.1186/1751-0473-8-13](https://link.springer.com/article/10.1186/1751-0473-8-13)
- ACMG Standards: Richards et al. (2015) *Genetics in Medicine* 17(5):405-24
- CPIC Guidelines: [cpicpgx.org](https://cpicpgx.org)
- Ensembl VEP: [ensembl.org/info/docs/tools/vep](https://www.ensembl.org/info/docs/tools/vep/index.html)
- ClinVar: [ncbi.nlm.nih.gov/clinvar](https://www.ncbi.nlm.nih.gov/clinvar/)
- gnomAD: [gnomad.broadinstitute.org](https://gnomad.broadinstitute.org/)

---

*This tutorial is part of ClawBio's open educational resources. Licensed under MIT.*